# 🎯 *Aprendizaje Automático en Negocios - Métricas, A/B Testing y Bootstrap* 🎯

---


## **🎓 Objetivos académicos**
* Construir un embudo de conversión a partir de múltiples fuentes de datos.
* Realizar un análisis exploratorio de comportamiento por segmento.
* Comparar tasas de conversión segmentadas por dispositivo.
* Proponer una prueba A/B fundamentada con datos.

---


In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as patches
from scipy.stats import ttest_ind



In [ ]:
sns.set(style="whitegrid", palette="pastel")


## 🧩 *Métricas generales de negocio*

**📈 Conceptos clave:**
* *Ingresos:* lo que entra 💰
* *Coste de bienes vendidos (COGS):* lo que cuesta producir 🏭
* *Beneficio bruto:* `ingresos - COGS`
* *Margen bruto:* `(beneficio bruto / ingresos)`
* *Gastos operativos:* renta, sueldos, marketing...
* *Beneficio operativo:* `beneficio bruto - gastos operativos`
* *Beneficio neto:* lo que realmente se gana (después de impuestos)
* *ROI:* `beneficio neto / inversión`
* *Conversión:* usuarios que completan una acción (ej. compra)
* *Embudos:* visualizan pasos como visitante → cliente

**💡 Intuición:**
Estas métricas son el puente entre el mundo de los datos y el negocio real. ¡No basta con tener un modelo preciso! Lo que importa es si mejora ingresos, conversión o rentabilidad.

### 🧪 Análisis de resultados de campañas de marketing

Eres parte del equipo de análisis de una empresa digital que ha implementado varias campañas de marketing para atraer usuarios y generar conversiones en su plataforma. Ahora cuentas con tres conjuntos de datos clave que te permitirán realizar un análisis integral del desempeño de las campañas:



1. **Interacciones y transacciones `orders`:**  
   - Cada registro representa una interacción de usuario que resultó en una transacción.
   - Columnas: `uuid_interaction` (ID de interacción), `transaction_ts` (fecha/hora), `items` (artículos comprados), `amount` (monto total).

2. **Costos de campañas `campaign_detaials` :**  
   - Información de costos fijos y variables para cada campaña.
   - Columnas: `id_campaña`, `costo_base_campaña`, `costo_por_dia_activo_campaña`, `costo_por_interaccion_campaña`.

3. **Visitas y conversiones de usuarios `visits`:**  
   - Cada registro representa una visita de usuario a la plataforma, asociada a una campaña.
   - Columnas: `uuid_user` (ID de usuario), `ts_visits` (fecha/hora), `campaign` (ID de campaña), `uuid_interaction` (ID de interacción), `convertion` (1 si hubo conversión, 0 si no).



#### Sugerencias de análisis de interés

- **Ingresos y artículos vendidos por campaña:** Suma el monto total y la cantidad de artículos vendidos asociados a cada campaña.
- **Rentabilidad de campañas:** Relaciona los ingresos y conversiones generados con los costos totales (base, por día y por interacción) para identificar campañas más rentables.
- **Eficiencia de inversión:** Calcula métricas como el costo por conversión, el ingreso por visita y el retorno de inversión (ROI) para cada campaña.

Con estos análisis podrás obtener una visión completa del desempeño de las campañas, identificar oportunidades de mejora y tomar decisiones informadas para optimizar futuras estrategias

In [ ]:
#carga de datos

visits =pd.read_csv("https://raw.githubusercontent.com/zyntonyson/bootcamp_ds_da/refs/heads/main/datasets/visits.csv")
orders =pd.read_csv("https://raw.githubusercontent.com/zyntonyson/bootcamp_ds_da/refs/heads/main/datasets/orders.csv")
campaigns_details =pd.read_csv("https://raw.githubusercontent.com/zyntonyson/bootcamp_ds_da/refs/heads/main/datasets/campaigns_details.csv")


**Ingresos  por campaña:**

In [ ]:
# Revenue por campaña y clientes
revenue_campaign=(
    orders
        .merge(visits[['uuid_interaction','campaign','uuid_user']].drop_duplicates(),on='uuid_interaction')
        .groupby(['campaign'],as_index=False)
        .agg(
            revenue = ('amount','sum'),
            clientes = ('uuid_user','nunique')
        )
        #.style.format({'revenue':'$ {:,.2f}'})
)

revenue_campaign

**Interacciones por campaña:**

In [ ]:

campaign_summary=(
    visits
        .assign(
            ts_visits = lambda df: pd.to_datetime(df.ts_visits) ,
            date_visits = lambda df: df.ts_visits.dt.date

        )
        .groupby(['campaign'],as_index=False)
        .agg(
            visitas = ('uuid_interaction','nunique'),
            rate_convertion = ('convertion','mean'),
            init_campaign = ('ts_visits','min'),
            end_campaign = ('ts_visits','max'),
        )
        .assign(
        days_elapsed_campaign= lambda df: (df.end_campaign - df.init_campaign).dt.days ,
        rate_convertion = lambda df: df.rate_convertion.round(3),
        convertions = lambda df: (df.rate_convertion*df.visitas).round(0)

    )
    .drop(columns=['init_campaign','end_campaign'])
)

campaign_summary

**Rentabilidad por campaña**

In [ ]:
def get_romi(row):
    # Costos individuales
    base = row['costo_base_campaña']
    costo_dias = row['costo_por_dia_activo_campaña'] * row['days_elapsed_campaign']
    costo_interacciones = row['costo_por_interaccion_campaña'] * row['visitas']
    
    # Costo total
    costo_total = base + costo_dias + costo_interacciones
    
    # Ingreso
    revenue = row['revenue']
    
    # ROMI
    if costo_total == 0:
        return float('nan')  # Venta organica
    return (revenue - costo_total) / costo_total

In [ ]:
#Costos-rentabilidad

campaign_results=(campaign_summary
    .merge(revenue_campaign)
    .merge(campaigns_details, left_on='campaign',right_on='id_campaña')
    .assign(
        ROMI = lambda df: df.apply(get_romi,axis=1)
    )
)

In [ ]:
campaign_results[['campaign', 'costo_base_campaña',  'days_elapsed_campaign', 'costo_por_dia_activo_campaña', 'costo_por_interaccion_campaña',  'visitas', 'revenue','ROMI']]

**Embudo de conversión**

In [ ]:
def draw_funnel(data_funnel: dict):
    """
    Dibuja un embudo de conversión como un triángulo invertido con etiquetas y tasas de conversión.

    Args:
        data (dict): etapas del embudo (de mayor a menor) con sus valores.
    """
    stages = list(data_funnel.keys())
    values = list(data_funnel.values())
    n = len(stages)

    fig, ax = plt.subplots(figsize=(7, 8))

    # Triángulo invertido
    bottom = [0.5, 0.0]
    top_left = [0.0, 1.0]
    top_right = [1.0, 1.0]

    triangle = patches.Polygon([top_left, top_right, bottom], closed=True,
                                edgecolor='black', facecolor='white', linewidth=2)
    ax.add_patch(triangle)

    # Líneas horizontales divisorias
    for i in range(1, n):
        y = i / n
        x1 = 0.5 - 0.5 * (1 - y)
        x2 = 0.5 + 0.5 * (1 - y)
        ax.plot([x1, x2], [1 - y, 1 - y], color='black')

    # Etiquetas de etapas y tasas de conversión
    for i in range(n):
        y_top = 1 - (i / n)
        y_bottom = 1 - ((i + 1) / n)
        y_text = y_bottom+ 97*(y_top - y_bottom) / 100



        # Porcentaje de conversión desde la etapa anterior
        if i > 0:
            prev = values[i - 1]
            curr = values[i]
            if prev != 0:
                rate = 100 * curr / prev
                ax.text(0.5, y_text, f"{stages[i].upper()}\n\n{values[i]:,.0f}\n({rate:.1f}%)", ha='center', va='top', fontsize=15)
        else:
            # Nombre y valor de la etapa
            ax.text(0.5, y_text, f"{stages[i].upper()}\n\n{values[i]:,.0f}", ha='center', va='top', fontsize=15)


    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    plt.title('Embudo de Conversión', fontsize=14, weight='bold') #<--- Aqui puedes cambiar el titulo
    plt.tight_layout()
    plt.show()

In [ ]:
funnel_data=(
    
    campaign_results
    #.query("campaign == 2")
    .rename(columns={'convertions':'conversiones'})
    [['visitas','conversiones','clientes']]
    .sum()
    .to_dict()
)

In [ ]:
draw_funnel(funnel_data)

---

## 🧪 *Experimentación en los negocios pruebas A/B*


Una prueba A/B compara dos versiones de algo para identificar cuál tiene mejor rendimiento en una métrica clave.

* *Grupo de control:* recibe lo habitual.
* *Grupo experimental:* recibe la nueva funcionalidad.
* *Hipótesis nula (H₀):* no hay diferencia.
* *Hipótesis alternativa (H₁):* sí hay diferencia.
* *Error tipo I:* falso positivo (rechazas H₀ cuando es cierta).
* *Error tipo II:* falso negativo (aceptas H₀ cuando es falsa).
* *Valor p:* ¿qué tan probable es ver esta diferencia si H₀ es cierta?




**🧪Objetivo del experimento:**  
Evaluar si una nueva versión de una página web (grupo B) mejora la tasa de conversión respecto a la versión actual (grupo A).

**Hipótesis estadísticas:**

- **H₀ (hipótesis nula):** No hay diferencia en la tasa de conversión entre los grupos A y B ($p_A = p_B$).
- **H₁ (hipótesis alternativa):** La tasa de conversión del grupo B es diferente a la del grupo A ($p_A \ne p_B$).




📄 **Diccionario de datos**

| Variable     | Tipo       | Descripción                                              |
|--------------|------------|----------------------------------------------------------|
| grupo        | categórica | Grupo asignado al usuario: 'A' (control) o 'B' (test)    |
| conversion   | binaria    | Resultado: 1 si convirtió, 0 si no convirtió             

In [ ]:
ab_test_landing_page_data=pd.read_csv("https://raw.githubusercontent.com/30lm32/ml-ab-testing/refs/heads/master/ab_data.csv")

In [ ]:
treatment = ab_test_landing_page_data.query("group =='treatment'")['converted']
control = ab_test_landing_page_data.query("group =='control'")['converted']
alpha = 0.05

In [ ]:
(
    ab_test_landing_page_data
        .groupby('group',as_index=False)
        .agg(
            rate =('converted','mean')
        )
        .round(3)
)

In [ ]:
stat,p_value = ttest_ind(treatment,control,alternative='greater')

print(f"Tenemos un {p_value=:.3f} entonces con {alpha=} ")
if p_value > alpha:
    print(f"No hay evidencia de que las tasas de conversión sean diferentes")
else:
    print(f"Hay evidencia de que las tasas de conversión sean diferentes")

---

## 🔁 *Intervalos de confianza via Bootstrapping*

**📊 Conceptos clave:**
* *Bootstrapping:* técnica de remuestreo con reemplazo para estimar incertidumbre (intervalos de confianza, valor p).
* *No asume distribución normal* → útil en datos reales.
* *Permite simular múltiples realidades* con solo una muestra.

In [ ]:
# Revenue por campaña y clientes
(revenue_campaign
    .assign(
        LTV = lambda df: df.revenue/df.clientes
    )
    .style.format({'revenue':'${:,}','LTV':'${:,.0f}'})
 
)

**I.C. 95% para LTV para Campaña 2**

>Un intervalo de confianza es un rango de valores donde probablemente se encuentra un parámetro poblacional, con cierta certeza estadística.

In [ ]:
n_bootstrap_samples = 1000
alfa = 0.05

In [ ]:
bootstrap_values= []
data_selected =(orders
        .merge(visits[['uuid_interaction','campaign']].drop_duplicates(),on='uuid_interaction')
        .query("campaign == 2")[['amount']])


for _ in range(n_bootstrap_samples):
        
    current_sample=(data_selected
        .sample(300,replace=True)
        .mean()
        .tolist()[0])
    bootstrap_values.append(current_sample)

bootstrap_values= pd.Series(bootstrap_values)
    

In [ ]:
print( f"el intervalo bootstrap al {100*(1-alfa):.0f}% de confianza  para  LTV de los clientes es entre {bootstrap_values.quantile(alfa/2):.2f} y {bootstrap_values.quantile(1-alfa/2):.2f}")

In [ ]:
sns.histplot(bootstrap_values)

>🔎 **Intuición:**  
Esto significa que, si repitiéramos el experimento muchas veces con diferentes muestras, el valor real del LTV estaría dentro de ese rango en el 95% de los casos.  
No asegura que el LTV esté exactamente ahí, pero sí que tenemos alta certeza estadística de que ese es el rango probable.  
En otras palabras: **el intervalo nos da una idea de la incertidumbre y la precisión de nuestra estimación**. Si el intervalo es estrecho, nuestra estimación es más precisa; si es amplio, hay más incertidumbre.

**I.C. 95% para proporción de clientes esperados por Campaña 2**


In [ ]:
n_bootstrap_samples = 10000
alfa = 0.05

In [ ]:
bootstrap_values=[]
data_selected=(
    visits
        .merge(orders, how='left',on='uuid_interaction')[['campaign','uuid_user','uuid_interaction','amount']]
        .drop_duplicates()


)
n_sample = 1000

for _ in range(n_bootstrap_samples):
        
    current_sample=(data_selected
        .sample(n_sample,replace=True)
        .query("campaign == 2 and amount.notnull()")[['uuid_user']]
        .nunique()
        .tolist()[0])
    bootstrap_values.append(current_sample/n_sample)

bootstrap_values= pd.Series(bootstrap_values)


In [ ]:
print( f"el intervalo bootstrap al {100*(1-alfa):.0f}% de confianza  para  LTV de los clientes es entre {bootstrap_values.quantile(alfa/2):.2f} y {bootstrap_values.quantile(1-alfa/2):.2f}")

---

## Antes de cerrar, me gustaría saber tu opinión: 😊✨

----

- 🧠 ¿Qué fue lo más útil o interesante que aprendiste hoy?
- 🤔 ¿Qué parte te resultó más difícil o te gustaría repasar?
- 🚀 ¿Qué potencial le encuentras a la actividad en tu futuro trabajo como analista de datos?

## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta el proyecto nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [Sprint 11](https://discord.com/channels/1081207584104656986/1270074296395497513).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal `#project` para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨